# Bench `parse_pdf` sur le corpus client

Charge `bench_parse_pdf_results_js.csv` (résultats du run de `parse_pdf` sur l'intégralité des PDFs partagés par Kezhan dans `data/`) et affiche les distributions clés.

## Ce que contient le bench

Le CSV à la racine du repo a été produit en exécutant `parse_pdf(pdf_path)` sur **tous les PDFs de `data/`** (corpus client : `CG contrats MRH`, `annuel_reports`, `insurance`, `cmo`, `nist`, `paper`, `reports`). Pour chaque PDF, on capture : `n_pages`, `source_tool`, `content_type`, `recommended_strategy`, distribution des `page_type`, nombre de pages à OCR-iser, temps de traitement.

**Objectif** : valider que `parse_pdf` tient sur le corpus réel sans erreur, et donner une vue agrégée du profil du corpus (combien de PDFs natifs, combien de mixtes, quels outils source détectés).

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('..') / 'bench_parse_pdf_results_js.csv')

print(f'Total PDFs : {len(df)}')
print(f"Total pages : {int(df['n_pages'].sum())}")
print(f"Pages needing OCR : {int(df['pages_needing_ocr'].sum())}")
print(f"Temps total : {df['parse_seconds'].sum():.1f}s ({df['parse_seconds'].sum()/60:.1f} min)")

errors = df[df['error'].fillna('').astype(str).str.strip() != '']
print(f"\nErreurs : {len(errors)} / {len(df)}")

print('\n=== Distribution content_type ===')
print(df['content_type'].fillna('(error)').value_counts().to_string())

print('\n=== Distribution source_tool ===')
print(df['source_tool'].fillna('(error)').value_counts().to_string())

print('\n=== Distribution recommended_strategy ===')
print(df['recommended_strategy'].fillna('(error)').value_counts().to_string())

print('\n=== Par corpus ===')
print(df.groupby('corpus').agg(n_pdfs=('path','count'), n_pages=('n_pages','sum'), avg_seconds=('parse_seconds','mean')).round(2).to_string())

Total PDFs : 71
Total pages : 6589
Pages needing OCR : 22
Temps total : 2586.5s (43.1 min)

Erreurs : 0 / 71

=== Distribution content_type ===
content_type
native    59
mixed     12

=== Distribution source_tool ===
source_tool
Adobe InDesign       29
Unknown              18
Microsoft Word        8
Ghostscript           7
Adobe PDF Library     2
Adobe Acrobat         2
Kofax                 1
Adobe Illustrator     1
PDF-XChange           1
ABBYY FineReader      1
iText                 1

=== Distribution recommended_strategy ===
recommended_strategy
use_native_extraction    59
per_page_routing         12

=== Par corpus ===
                 n_pdfs  n_pages  avg_seconds
corpus                                       
CG contrats MRH      11      809        26.09
annuel_reports       42     4424        43.32
cmo                   7      435        28.14
insurance             6      609        28.63
nist                  2       87         8.62
paper                 2       37        11.40